
# Steps for Conducting an A/B Test

This notebook describes the steps to conduct an A/B test and analyze the results effectively. The structured approach provides a clear understanding of the statistical concepts involved in A/B testing, including the distribution of data, expected values, and the process of hypothesis testing.


# A/B Testing in Practice:

Suppose a company wants to improve its webpage to increase user sign-ups. To determine which version is more effective, it uses A/B testing — a randomized experiment comparing the old webpage (control group, A) and the new webpage (test group, B) through statistical hypothesis testing.








## Steps for conducting the A/B Testing Experiment on the website

1. **Hypothesis**: Formulate a hypothesis, such as "the new webpage design will increase the sign-up rate by 10% compared to the old design."
2. **Sample Size**: Calculate the required sample size using power analysis to ensure statistical significance.
3. **Randomization**: Assign users randomly to either the control or variant group to avoid bias.
4. **Monitoring**: Track the defined metrics and check for any data quality issues during the test period.
5. **Statistical Test**: Use a **z-test** or **Chi-square test** to compare the proportions between the control and variant groups.
6. **Conclusion Based on p-value**: If the p-value is less than or equal to the significance level (commonly **0.05**), reject the null hypothesis.



In [2]:
!pip install --upgrade scipy


##1. Hypothesis

In [13]:
# Define the hypothesis
hypothesis = """
We hypothesize that the new webpage design will increase the sign-up rate by 10% compared to the old design."""

print(hypothesis)



We hypothesize that the new webpage design will increase the sign-up rate by 10% compared to the old design.


## 2. Sample Size


We use power analysis to calculate the required sample size for the A/B test to ensure that there is enough statistical power to detect a significant difference if one exists. Here we'll use`TTestIndPower` class to perform the power analysis, and `solve_power` method to calculate required sample size with params:

- effect_size: expected difference in sign-up rate between the two groups (here 10% increase).
-alpha: significance level (0.05).
- power: desired power of the test (0.8).
-ratio: ratio of sample sizes in the two groups (1 for equal sizes).


In [19]:
# Calculate required sample size using power analysis
from statsmodels.stats.power import TTestIndPower

# Parameters
effect_size = 0.1  # 10% increase in sign-up rate
alpha = 0.05
power = 0.8
ratio = 1  # Equal sample sizes in both groups

# Calculate sample size
analysis = TTestIndPower()
sample_size = analysis.solve_power(effect_size=effect_size, nobs1=None, alpha=alpha, power=power, ratio=ratio)

print(f"Required sample size per group: {int(sample_size)}")


Required sample size per group: 1570


## 3. Randomize Traffic

We randomize the assignment of users to the control (A) and variant (B) groups to ensure unbiased results.

Using the condition`random.random() < 0.5`:

- If the random number is less than 0.5, the function returns "A".

- If the random number is 0.5 or greater, the function returns "B".


In [20]:
import random

def randomize_traffic(user_id, sample_size):
    return "A" if random.random() < 0.5 else "B"

# Example usage
user_ids = list(range(1, int(sample_size * 2) + 1))
assignments = [randomize_traffic(user_id, sample_size) for user_id in user_ids]
print(f"Traffic assignments: {assignments}")


Traffic assignments: ['A', 'B', 'A', 'A', 'A', 'B', 'A', 'B', 'A', 'A', 'B', 'B', 'B', 'A', 'A', 'A', 'B', 'B', 'B', 'B', 'A', 'A', 'B', 'B', 'A', 'A', 'A', 'A', 'B', 'A', 'B', 'B', 'B', 'A', 'A', 'B', 'B', 'A', 'A', 'B', 'B', 'B', 'B', 'A', 'B', 'A', 'B', 'B', 'B', 'A', 'B', 'B', 'A', 'B', 'A', 'B', 'B', 'A', 'A', 'B', 'B', 'B', 'A', 'B', 'B', 'A', 'A', 'A', 'B', 'A', 'B', 'B', 'B', 'A', 'A', 'B', 'A', 'B', 'B', 'B', 'B', 'A', 'A', 'A', 'A', 'A', 'B', 'B', 'A', 'B', 'A', 'A', 'A', 'B', 'A', 'B', 'B', 'A', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'A', 'B', 'B', 'A', 'A', 'A', 'A', 'A', 'B', 'A', 'A', 'B', 'A', 'A', 'A', 'B', 'B', 'A', 'B', 'A', 'A', 'B', 'B', 'A', 'B', 'A', 'A', 'A', 'B', 'B', 'A', 'B', 'B', 'B', 'B', 'A', 'B', 'A', 'A', 'B', 'A', 'A', 'B', 'A', 'B', 'B', 'B', 'A', 'A', 'B', 'A', 'B', 'A', 'B', 'B', 'A', 'A', 'B', 'B', 'A', 'B', 'B', 'A', 'A', 'B', 'B', 'B', 'B', 'A', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'B', 'A', 'B'

 ### Create a Data Frame

with user IDs, their assigned groups, and whether they signed up. The signed_up column simulates the outcome of the user interaction (0 for no sign-up, 1 for sign-up).

In [21]:
# Simulate data collection
import pandas as pd

data = {
    'user_id': user_ids,
    'version': assignments,
    'signed_up': [random.choice([0, 1]) for _ in range(len(user_ids))]
}

df = pd.DataFrame(data)
print(df.head())


   user_id version  signed_up
0        1       A          1
1        2       B          1
2        3       A          1
3        4       A          0
4        5       A          0


##4.Monitor Metrics

Track the defined metrics and check for any data quality issues. This includes:

### Tracking Metrics:

Continuously monitor the conversion rates for both groups to ensure the test is progressing as expected. This can be done using dashboards or automated reports that update in real-time. For example, you can use tools like Google Analytics, Mixpanel, or custom dashboards built with Python libraries like Plotly or Dash.

In [22]:
import plotly.express as px

# Assuming df is your DataFrame with columns 'version' and 'signed_up'
conversion_rates = df.groupby('version')['signed_up'].mean()
print(conversion_rates)

# Visualize the conversion rates
fig = px.bar(conversion_rates, title='Conversion Rates by Version')
fig.show()


version
A    0.498116
B    0.520336
Name: signed_up, dtype: float64


### Data Quality Checks:

Verify that the data being collected is accurate and complete. Look for any anomalies or inconsistencies that could affect the results. This includes checking for missing values, outliers, and ensuring that the data collection process is functioning correctly.

In [23]:
# Check for missing values
missing_values = df.isnull().sum()
print(missing_values)

# Check for outliers in the 'signed_up' column
import numpy as np
outliers = np.where(df['signed_up'] > 1)
print(outliers)


user_id      0
version      0
signed_up    0
dtype: int64
(array([], dtype=int64),)


### Interim Analysis:

Conduct interim analyses to check if the test can be stopped early due to significant results or if adjustments are needed. This involves performing statistical tests at predefined intervals to see if the results are already conclusive. Tools like Sequential Probability Ratio Tests (SPRT) or Bayesian methods can be used for this purpose.

## 5. Statistical Test - Z-Test

In [25]:
# Perform z-test for proportions
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

# Data
kA = df[df['version'] == 'A']['signed_up'].sum()  # Number of conversions (successes) for the old webpage (control group A)
kB = df[df['version'] == 'B']['signed_up'].sum()  # Number of conversions (successes) for the new webpage (variant group B)

# Total users in each group
nA = df[df['version'] == 'A'].shape[0]  # Total users shown the old webpage (A)
nB = df[df['version'] == 'B'].shape[0]  # Total users shown the new webpage (B)

# Calculate conversion rates
pA = kA / nA  # Conversion rate for the old webpage (A)
pB = kB / nB  # Conversion rate for the new webpage (B)

# Perform a z-test for proportions
z_score, p_value = proportions_ztest(
    [kA, kB],
    [nA, nB]
)

# Output results
print(f"Control Conversion Rate (pA): {pA:.2%}")  # Print conversion rate for control group
print(f"Variant Conversion Rate (pB): {pB:.2%}")   # Print conversion rate for variant group
print(f"Z-Score: {z_score:.2f}")                    # Print z-score
print(f"P-Value: {p_value:.4f}")                   # Print p-value

# Conclusion based on p-value
alpha = 0.05  # Significance level
if p_value <= alpha:
    print("Reject the null hypothesis: The new feature has a significant effect.")
else:
    print("Fail to reject the null hypothesis: No significant effect observed.")


Control Conversion Rate (pA): 49.81%
Variant Conversion Rate (pB): 52.03%
Z-Score: -1.25
P-Value: 0.2130
Fail to reject the null hypothesis: No significant effect observed.


## 5. Chi-square test for proportions

In [26]:
from scipy.stats import chi2_contingency


# Create a contingency table
contingency_table = np.array([[kA, nA - kA], [kB, nB - kB]])

# Perform a Chi-square test
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

# Output results
print(f"Control Conversion Rate (pA): {kA / nA:.2%}")  # Print conversion rate for control group
print(f"Variant Conversion Rate (pB): {kB / nB:.2%}")   # Print conversion rate for variant group
print(f"Chi-Square Statistic: {chi2:.2f}")              # Print Chi-square statistic
print(f"P-Value: {p_value:.4f}")                       # Print p-value

# Conclusion based on p-value
alpha = 0.05  # Significance level
if p_value <= alpha:
    print("Reject the null hypothesis: The new feature has a significant effect.")
else:
    print("Fail to reject the null hypothesis: No significant effect observed.")


Control Conversion Rate (pA): 49.81%
Variant Conversion Rate (pB): 52.03%
Chi-Square Statistic: 1.46
P-Value: 0.2264
Fail to reject the null hypothesis: No significant effect observed.


### Understanding the Distribution of Data

The data for each group (A/B) can be modeled as a series of **Bernoulli trials**, where each trial has two possible outcomes:
- **1** (converted)
- **0** (not converted)

**The Mean of Bernoulli Distributions:**

The expected value of Bernoulli random variables represents the true conversion rates:

$ E(X_A) = p_A \times 1 + (1 - p_A) \times 0 = p_A $

$E(X_B) = p_B \times 1 + (1 - p_B) \times 0 = p_B $

The sample mean serves as the **maximum likelihood estimator (MLE)** of \( p \) based on a random sample. Thus, the signing-up rates \( p_A \) and \( p_B \) are the sample means for each group.

For samples larger than 30, the distribution of the sample means is approximately normally distributed around the true mean, with a standard deviation equal to the **standard error** of the mean.

To assess how far off the estimates of $ p_A$ or $ p_B $ might be from the true means, we compute the **standard error**. This measure provides insight into the variability of the sample means and helps quantify the uncertainty in our estimates.

### Statistical Inference

We aim to determine if the differences in signing-up rates between groups A and B are statistically significant. This means we want to establish that the differences are likely real, repeatable, and not due to random chance.

To do this, we start by assuming the **null hypothesis** $ H_0$: "The true success rates of the two webpages are equal." This can be mathematically represented as:

- $ H_0: p_A = p_B $
- This implies that the difference in success rates $ d_{AB} = 0 $.

By testing this hypothesis, we can evaluate whether the observed differences in conversion rates are statistically significant or if they could have occurred by random chance.

The likelihood framework underpins this analysis, as the Z-test for proportions is based on the likelihood of observing the data given the parameters. The maximum likelihood estimates of $p_A $ and $ p_B $ are derived from the observed conversion rates, allowing us to assess the significance of the differences in a statistically rigorous manner.

We also perform a Chi-square test for proportions. The `chi2_contingency` function from the `scipy.stats` module performs the Chi-square test on this contingency table.


## 6. Conclusion

Given that the p-value (0.2130) is greater than the significance level (commonly 5% or 0.05), we fail to reject the null hypothesis.

That means, the new webpage design did not show a statistically significant increase in the sign-up rate compared to the old design.

